# Meta DP ++ Algorithm Tomson Sampling (MTS++) based agents

> Agents utelizing the Meta DP TS ++ based approach for Dynamic pricing and learning problems from https://pubsonline.informs.org/doi/10.1287/mnsc.2021.4071

In [ ]:
#| default_exp agents.dynamic_pricing.MTS

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

from abc import ABC, abstractmethod
from typing import Union, Optional, List
import numpy as np
import joblib
import os
import statsmodels.api as sm
from ddopai.agents.dynamic_pricing.utils import GLMLink
from ddopai.envs.base import BaseEnvironment
from ddopai.agents.dynamic_pricing.mushroom_rl import PricingMushroomBaseAgent
from mushroom_rl.core import Agent
from ddopai.utils import MDPInfo
from ddopai.agents.obsprocessors import FlattenTimeDimNumpy
from ddopai.envs.actionprocessors import ClipAction


In [ ]:
#| export
class MTSPolicy():
    def __init__(self,
                 environment_info: MDPInfo, # Environment metadata including observation and action spaces
                 sigma: float, # Known standard deviation of demand noise 
                 lambda_e: float, # Regularization parameter for exploration
                 exploration:float=1e-2, # Exploration parameter
                 c_0:float = 1, # Regularization parameter for the price function
                 c_2_const:float = 1, # Regularization parameter for the price function
                 x_max:float = 1, # Maximum value of the state space
                 N: int = 400, 
                 obsprocessors: Optional[List[object]] = None, # Initial exploratory prices used in the first few epochs
                 actionprocessors: Optional[List[object]] = None, 
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 price_function = None, # Function that computes optimal prices given parameters and observations
                 g=None):
        self.env_info = environment_info
        self.actionprocessors = actionprocessors if actionprocessors else []
        self.obsprocessors = obsprocessors if obsprocessors else []
        
        self.price_function = price_function
        self.sigma = sigma
        self.lambda_e = lambda_e
        self.p_min = environment_info.action_space.low
        self.p_max = environment_info.action_space.high
        self.c_0 = c_0
        self.c_1 = c_0/(np.sqrt(1+self.p_max)**2*x_max)
        self.c_2 = c_2_const/self.c_1
        self.ex_prices = np.array(ex_prices) if ex_prices is not None else np.array([0, self.p_max])
        self.T = environment_info.horizon
        self.d = environment_info.observation_space['features'].shape[0]
        self.N = N
        self.g = g

        self.th_hat_store = []
        self.correction = np.zeros((2*self.d, 2*self.d))

        self.th_hat = np.zeros(2*self.d)
        self.sig_mpdp = np.eye(2*self.d)
        self.f = np.zeros(2*self.d)
        
        self.X = np.empty((0, 2*self.d))
        self.Y = np.empty((0, 1))

        self.t_e = max(int(np.ceil(4 * np.log(self.d * self.T * self.N ))), 2*self.lambda_e/self.c_0)
        self.N_0 = int((self.c_2*self.d)**2/(self.lambda_e*2))
        self.t = 0
        self.actionprocessors.append(ClipAction(environment_info.action_space.low, environment_info.action_space.high))
        self.mode = "train"
    
    def draw_action(self, observation: np.ndarray):
        if self.t < self.t_e:
                price = self.ex_prices[self.t % len(self.ex_prices)]
        else:
            X = observation['features']

            th_dot = np.random.multivariate_normal(self.th_hat, self.sig_mpdp)
            price = self.price_function(X, th_dot[:self.d], th_dot[self.d:])
                
        for processor in self.actionprocessors:
            price = processor(price)
        
        return np.array(price)
    
    def fit(self, X, Y, action):
        self.t += 1
        x_action = np.concatenate([X, X * action])
        self.X = np.vstack([self.X, x_action])
        self.Y = np.vstack([self.Y, Y])
        self.parameter_update(x_action, Y)

    def parameter_update(self, x_action, Y):
        reward = Y.flatten()  # Assuming single reward per observation
        vector = x_action.flatten()

        if self.t >= self.t_e:
            self.f += reward * vector
            self.sig_mpdp -= (self.sig_mpdp @ np.outer(vector, vector) @ self.sig_mpdp) / (1 + vector.T @ self.sig_mpdp @ vector)
            self.th_hat = self.sig_mpdp @ self.f

    
    def update_env(self, env):
        if self.X.size > 0:
            current_th_hat = np.linalg.inv(self.X.T @ self.X) @ self.X.T @ self.Y
            self.th_hat_store.append(current_th_hat)
            self.correction += np.linalg.pinv(self.X.T @ self.X)

            i = len(self.th_hat_store)
            if i >= self.N_0:
                dumb = np.hstack(self.th_hat_store).T
                self.th_hat = np.mean(dumb, axis=0)

                cov_dumb = np.cov(dumb, rowvar=False)
                self.sig_mpdp = ((i - 1) * cov_dumb / (i - 2) - 
                                0.8 * self.sigma * self.correction / (i - 1) +
                                0.5 * self.d * np.eye(2*self.d) / np.sqrt(i))

                self.f = np.linalg.inv(self.sig_mpdp) @ self.th_hat

        self.environment_info = env.mdp_info
        self.X = np.empty((0, 2*self.d))
        self.Y = np.empty((0, 1))
        self.t = 0
        
    def reset(self):
        pass


In [ ]:
#| export
class MTSCoreAgent(Agent):

    """
    Base class for TS agents.
    """

    def __init__(self,
                 environment_info: MDPInfo,
                 sigma: float, # Known standard deviation of demand noise
                 lambda_e: float, # Regularization parameter for exploration
                 exploration:float, # Exploration parameter
                 c_0: float = 0, # Regularization parameter for the price function
                 c_2_const:float = 1, # Regularization parameter for the price function
                 x_max:float = 1, # Maximum value of the state space
                 N: int = 400,
                 obsprocessors: Optional[List[object]] = [],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        
        policy = MTSPolicy(environment_info=environment_info, sigma=sigma, lambda_e=lambda_e, exploration=exploration, c_0=c_0, c_2_const=c_2_const, x_max=x_max, N=N, obsprocessors=obsprocessors, actionprocessors=actionprocessors,agent_name=agent_name, ex_prices=ex_prices, price_function=price_function, g=g)
        self.agent_name = agent_name
        super().__init__(environment_info, policy)
        
    def fit(self, dataset, **kwargs):
        X = dataset[0][0]["features"]
        Y = kwargs["demand"][0]
        action = dataset[0][1]
        self.policy.fit(X, Y, action)
    def update_env(self, env):
        self.policy.update_env(env)


In [ ]:
#| export
class MTSAgent(PricingMushroomBaseAgent):
    """
    Wrapper class for TSCoreAgent to interact with MushroomRL.
    """
    def __init__(self,
                 environment_info: MDPInfo,
                 sigma: float, # Known standard deviation of demand noise
                 lambda_e: float, # Regularization parameter for exploration
                 exploration:float, # Exploration parameter
                 c_0: float = 0, # Regularization parameter for the price function
                 c_2_const:float = 1, # Regularization parameter for the price function
                 x_max:float = 1, # Maximum value of the state space
                 N: int = 400,
                 obsprocessors: Optional[List[object]] = [],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        self.agent = MTSCoreAgent(environment_info=environment_info, sigma=sigma, lambda_e=lambda_e, 
                                  exploration=exploration, c_0=c_0, c_2_const=c_2_const, x_max=x_max, N=N, obsprocessors=obsprocessors, actionprocessors=actionprocessors, 
                                  agent_name=agent_name, ex_prices=ex_prices, price_function=price_function, g=g)
        super().__init__(environment_info=environment_info, obsprocessors=obsprocessors, agent_name=agent_name)
        
    def update_env(self, env: object):
        """ Update the environment specific parameters of the agent """
        self.agent.update_env(env)
